In [ ]:
# 형상 DOE for
    # 응력해석  
    # 온도/해석종류 
    # 무부하해석
# 전류/위상각 for
    # 해석
    # 데이터 추출
 # 다른 해석 FEA 추출   
    # 열해석


# 1) Environment / Imports (for debugging)

In [2]:
# 1) Environment / Imports (for debugging)
import pathlib
import sys

import matplotlib.pyplot as plt
import numpy as np


# Ensure repo root (folder containing 'tools/' or legacy 'tool/') is on sys.path
repo_root = pathlib.Path.cwd().resolve()
while not ((repo_root / "tools").exists() or (repo_root / "tool").exists()) and repo_root != repo_root.parent:
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))
print("repo_root:", repo_root)

# Force reload of local package during iterative edits


# Import utilities extracted from the notebook
from tools.motorCAD.pyMCAD import (
    get_magnetic_data,
    get_magnetic_data_from_file,
    get_magnetic_timeseries_from_file,
    interactive_magnetic_plot,
    interactive_magnetic_quiver,
    interactive_b_locus_field_plot,
    get_element_loss_fields,
    interactive_loss_fields_plot,
    mcad_default_export_dir,
    mcad_make_temp_txt_path,
    find_latest_mes,
    )
import importlib
import tools.motorCAD.pyMCAD as _pyMCAD
importlib.reload(_pyMCAD)
# 1.5) Enable zoomable Matplotlib backend (ipympl)
try:
    import ipympl  # noqa: F401
    get_ipython().run_line_magic("matplotlib", "widget")
    import matplotlib
    print("matplotlib backend:", matplotlib.get_backend())
except Exception as e:
    print("ipympl/widget backend not available; using default backend. Error:", e)

import os
import matplotlib.pyplot as plt
import ansys.motorcad.core as pymotorcad
mc = pymotorcad.MotorCAD(open_new_instance=False)
# mc.initialise_tab_names()    # 꼭 필요한지

repo_root: D:\KDH\NvidiaNemo\eMach
matplotlib backend: widget


In [ ]:
# Find *.mot files (uses repo-local pyAEDT utilities)
from tools.pyutils.import_path import ensure_repo_root_on_path

_ = ensure_repo_root_on_path()
from pyAEDT.aedt_file_utils import find_files

DOEDir = r"E:\\KDH\\AdaptiveTemplate\\TestCAD1\\optiSLangExport\\ExportedProj.opd\\Sensitivity__FromPara"
fileList = find_files(DOEDir, ".mot")

# print(f"Found {len(fileList)} .mot files")
# fileList[:5]  # preview

# Parametric Sweep 설정

In [ ]:
# Build sweep points and Motor-CAD case dictionaries
from tools.pyutils.import_path import ensure_repo_root_on_path

_ = ensure_repo_root_on_path()
from tools.pyutils.sweep import mkIpkPhaseMap, to_mcad_cases

ipeak_steps = 6
phase_steps = 8

ipeaks, phases, sweep_points = mkIpkPhaseMap(ipeak_steps, phase_steps)
cases = to_mcad_cases(sweep_points, ipeak_key="PeakCurrent", phase_key="PhaseAdvance")


In [ ]:
# 2) Export + Parse (time series)
from pathlib import Path
from datetime import datetime

# Output file base name
out_dir = mcad_default_export_dir(mc)
base_filename = Path(out_dir) / "MagTransient.txt"
print("Output dir:", out_dir)
print("Base output file:", base_filename)

# Export settings
DO_EXPORT = True
first_step = 1
final_step = 45

def _unique_filename_if_exists(path: Path) -> Path:
    """If 'path' exists, return a new path with a timestamp suffix."""
    if not path.exists():
        return path
    stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    return path.with_name(f"{path.stem}_{stamp}{path.suffix}")


In [ ]:
# List available .mes results by type (for the CURRENTLY OPEN .mot)
# NOTE: Outputs are saved under export dir you pass to process_fea_result_from_mes,
# or default is the folder containing the open .mot file (mcad_default_export_dir).
from tools.motorCAD.pyMCAD import list_mes_files

mes_by_kind = list_mes_files(mc)
for kind, paths in mes_by_kind.items():
    if not paths:
        continue
    print(f"{kind}: {len(paths)}")
    for p in paths[:5]:
        print("  ", p)

# export

In [ ]:
# Batch export (CURRENTLY OPEN .mot 기준): FEResultsData의 모든 .mes를 종류별로 처리해서 GIF/SVG로 저장
# - OnLoadTorque_result_* : transient => Mag_b.gif, Mag_a.gif, Mag_j.gif
# - StaticLoad/StaticOC : Mag_b.svg, Mag_a.svg, Mag_j.svg
# - OnLoadLoss : Loss_Pt.svg, Loss_Phys.svg, Loss_Pj.svg, Loss_Peddy.svg (등)
# - Thermal : Thermal_t.svg, Thermal_g.svg, Thermal_q.svg
# - Centrifugal(stress) : Stress_svm.svg, Stress_sp1.svg, Stress_sp2.svg, Stress_sx.svg, Stress_sy.svg, Stress_txy.svg
# - Cogging : Cogging_*.svg
#
# 추가: OnLoadTorque transient는 txt를 이미 파싱하므로, 원하면 바로 .h5로도 저장 가능
from pathlib import Path
import importlib

PLOT_MODE = "interactive"   # "gif" => OnLoadTorque는 GIF, snapshot들은 SVG | "none" => plot 저장은 안 하고 export만 수행
GIF_FPS = 6
EXPORT_SUBDIR = "postproc"  # FEResultsData 아래에 저장할 폴더

# Transient export step range (OnLoadTorque일 때만 의미 있음)
FIRST_STEP = 1
FINAL_STEP = 45

# Parsed transient(ts) -> HDF5 저장 여부 (h5py 필요)
EXPORT_MAG_H5 = True

# H5 mesh coords 저장 방식:
# - "by_step": 모든 node를 step별로 저장 (정확하지만 용량 큼)
# - "by_step_moving_nodes": rotor + a2/a3/a4 같은 회전 영역 node만 step별 저장 (용량 절감)
MAG_H5_MESH_COORDS = "by_step_moving_nodes"

# 이미 mot가 열려있다고 가정 (테스트용). 필요하면 아래를 직접 해제해서 사용.
# mc.load_from_file(r"...")
mc.display_screen(r"E-Magnetics;FEA")

# Reload local helpers (tools/ 수정사항 즉시 반영)
import tools.motorCAD.pyMCAD.magnetic as _magnetic
import tools.motorCAD.pyMCAD.fea_workflow as _fea_workflow
import tools.motorCAD.pyMCAD.results as _mcad_results
importlib.reload(_magnetic)
importlib.reload(_mcad_results)
importlib.reload(_fea_workflow)

from tools.motorCAD.pyMCAD import list_mes_files
from tools.motorCAD.pyMCAD.fea_workflow import process_fea_result_from_mes

mes_by_kind = list_mes_files(mc)
total = sum(len(v) for v in mes_by_kind.values())
print(f"Found {total} .mes for current MOT")


In [ ]:
mes_by_kind

In [ ]:
paths

In [ ]:

for kind, paths in mes_by_kind.items():
    if not paths:
        continue

    print(f"\n--- {kind}: {len(paths)}")
    for mes_path in paths:
        mes_path = Path(mes_path)
        out_dir = mes_path.parent / EXPORT_SUBDIR
        out_dir.mkdir(parents=True, exist_ok=True)

        result = process_fea_result_from_mes(
            mc,
            mes_path=mes_path,
            plot_mode=PLOT_MODE,
            out_dir=out_dir,
            first_step=int(FIRST_STEP),
            final_step=int(FINAL_STEP),
            point_size=2,
            gif_fps=int(GIF_FPS),
            export_magnetic_h5=bool(EXPORT_MAG_H5),
            mag_h5_mesh_coords=str(MAG_H5_MESH_COORDS),
        )
        print("mes:", mes_path.name)
        print("  saved to:", out_dir)
        if getattr(result, "mag_h5_path", None):
            print("  mag_h5_path:", result.mag_h5_path)
        if getattr(result, "mag_gif_paths", None):
            print("  mag_gif_paths:", result.mag_gif_paths)
        if getattr(result, "mag_svg_paths", None):
            print("  mag_svg_paths:", result.mag_svg_paths)
        if getattr(result, "loss_svg_paths", None):
            print("  loss_svg_paths:", result.loss_svg_paths)
        if getattr(result, "thermal_svg_paths", None):
            print("  thermal_svg_paths:", result.thermal_svg_paths)
        if getattr(result, "stress_svg_paths", None):
            print("  stress_svg_paths:", result.stress_svg_paths)
        if getattr(result, "cogging_svg_path", None):
            print("  cogging_svg_path:", result.cogging_svg_path)


# 2 .(From exported Mag_*.h5) Make GIF (timeseries) + SVG (static snapshot or selected step)


In [ ]:
# (From by_step_moving_nodes H5) Interactive plots
#
# This cell finds an H5 exported with mesh_coords_mode='by_step_moving_nodes',
# loads it lazily as a time series, then calls the existing interactive plot helpers.

import importlib
from pathlib import Path

import tools.motorCAD.pyMCAD as _pyMCAD
import tools.motorCAD.pyMCAD.magnetic as _magnetic
importlib.reload(_magnetic)
importlib.reload(_pyMCAD)

from tools.motorCAD.pyMCAD import (
    get_magnetic_timeseries_from_file,
    interactive_magnetic_plot,
    interactive_magnetic_quiver,
    list_mes_files,
    diagnose_magnetic_h5_mesh_motion,
    inspect_magnetic_timeseries_h5,
    interactive_b_locus_field_plot,
 )

# Reuse globals from Cell 10 if present
EXPORT_SUBDIR_LOCAL = str(globals().get("EXPORT_SUBDIR", "postproc"))
POINT_SIZE_LOCAL = float(globals().get("POINT_SIZE", 2))
CMAP_LOCAL = str(globals().get("CMAP", "jet"))
QUANTITY_LOCAL = str(globals().get("GIF_QUANTITY", "b")).lower().strip()

# 1) Collect Mag_*.h5 candidates under each results folder's postproc directory
h5_files: list[Path] = []
mes_by_kind = list_mes_files(mc)
for kind, paths in mes_by_kind.items():
    for mes_path in paths:
        mes_path = Path(mes_path)
        cand_dir = mes_path.parent / EXPORT_SUBDIR_LOCAL
        if cand_dir.exists():
            h5_files.extend(sorted(cand_dir.glob("Mag_*.h5")))
h5_files = sorted(set(h5_files))
print("Found Mag_*.h5:", len(h5_files))

if not h5_files:
    raise FileNotFoundError("No Mag_*.h5 found. Run the export cell with EXPORT_MAG_H5=True first.")

# 2) Prefer the most recently modified by_step_moving_nodes timeseries H5
candidates: list[Path] = []
for p in h5_files:
    try:
        info = inspect_magnetic_timeseries_h5(p)
        fmt = str(info.get("attrs", {}).get("format", ""))
        mcm = str(info.get("attrs", {}).get("mesh_coords_mode", ""))
        if ("timeseries" in fmt.lower()) and (mcm.lower() == "by_step_moving_nodes"):
            candidates.append(p)
    except Exception:
        continue

if candidates:
    h5_path = max(candidates, key=lambda pp: pp.stat().st_mtime)
else:
    # fallback: most recent timeseries H5
    ts_cands: list[Path] = []
    for p in h5_files:
        try:
            info = inspect_magnetic_timeseries_h5(p)
            fmt = str(info.get("attrs", {}).get("format", ""))
            if "timeseries" in fmt.lower():
                ts_cands.append(p)
        except Exception:
            continue
    if not ts_cands:
        raise RuntimeError("No timeseries Mag_*.h5 found.")
    h5_path = max(ts_cands, key=lambda pp: pp.stat().st_mtime)

print("Using:", h5_path)
motion = diagnose_magnetic_h5_mesh_motion(h5_path)
print("mesh_coords_mode:", motion.get("mesh_coords_mode"))
print("moving_node_count:", motion.get("moving_node_count"))
print("moving_reg_codes:", motion.get("moving_reg_codes_labeled") or motion.get("moving_reg_codes"))
print("conclusion:", motion.get("conclusion"))

# 3) Load time series lazily and plot interactively
ts = get_magnetic_timeseries_from_file(h5_path, key="time_index")
print("steps:", len(ts.steps), "range=", (ts.steps[0], ts.steps[-1]))

# Interactive scalar field plot with step slider
interactive_magnetic_plot(ts, quantity=QUANTITY_LOCAL, s=POINT_SIZE_LOCAL, cmap=CMAP_LOCAL)

# Optional: quiver plot (can be heavy). Increase stride if slow.
interactive_magnetic_quiver(ts, stride=2, scale=1, normalize=False)

# Optional: B-locus plot
try:
    interactive_b_locus_field_plot(ts)
except Exception as e:
    print("interactive_b_locus_field_plot skipped:", e)

In [ ]:
# 3) Debug: Compare TXT vs H5 MagneticRegions (focus: reg_code=94 a2 partial rotation)

from pathlib import Path
import numpy as np

from tools.motorCAD.pyMCAD.magnetic import export_magnetic_txt

REG_CODE = 94  # a2
STEP0 = 0       # index into the time series (0 = first)
STEP1 = -1      # index into the time series (-1 = last)

# Optional: manually set a specific TXT to compare (recommended if there are many .txt)
TXT_PATH_OVERRIDE = None  # e.g. r"E:\\...\\MagTransient.txt"

# Inputs: reuse h5_path from Cell 12 if present
H5_PATH = Path(globals().get("h5_path", "")) if str(globals().get("h5_path", "")).strip() else None
if not H5_PATH or not H5_PATH.exists():
    raise FileNotFoundError("H5 path not found. Run Cell 12 first (it sets variable h5_path).")

# 1) Pick / generate a TXT to compare against
TXT_PATH = None
if TXT_PATH_OVERRIDE:
    TXT_PATH = Path(TXT_PATH_OVERRIDE)
else:
    try:
        cand_dir = H5_PATH.parent
        txt_cands = sorted(list(cand_dir.glob("*.txt")))
        # prioritize likely files
        txt_cands = sorted(
            txt_cands,
            key=lambda p: (
                0 if "mag" in p.name.lower() else 1,
                0 if "trans" in p.name.lower() else 1,
                -p.stat().st_mtime,
            ),
        )
        if txt_cands:
            TXT_PATH = txt_cands[0]
    except Exception:
        TXT_PATH = None

if TXT_PATH is None or not TXT_PATH.exists():
    # Export a compare TXT (uses current Motor-CAD instance `mc`)
    compare_txt = H5_PATH.with_name(H5_PATH.stem + "_compare_export.txt")
    print("No nearby TXT found; exporting:", compare_txt)
    export_magnetic_txt(
        mc,
        first_step=int(globals().get("FIRST_STEP", 1)),
        final_step=int(globals().get("FINAL_STEP", 45)),
        filename=compare_txt,
    )
    TXT_PATH = compare_txt

print("H5:", H5_PATH)
print("TXT:", TXT_PATH)

# 2) Load timeseries from both sources
ts_h5 = get_magnetic_timeseries_from_file(H5_PATH, key="time_index")  # key ignored for H5 adapter
steps_h5 = ts_h5.steps
print("H5 steps[0:5]...:", steps_h5[:5], "len=", len(steps_h5))

# TXT: try to pick a key that matches H5 step keys, otherwise fall back to index-based alignment
def _load_txt_best_effort(path: Path, h5_steps: list[int]):
    ts_a = get_magnetic_timeseries_from_file(path, key="time_index")
    steps_a = ts_a.steps
    if set(steps_a).intersection(set(h5_steps)):
        return ts_a, "time_index"
    ts_b = get_magnetic_timeseries_from_file(path, key="solution")
    steps_b = ts_b.steps
    if set(steps_b).intersection(set(h5_steps)):
        return ts_b, "solution"
    # no match: return time_index but mark mismatch
    return ts_a, "time_index(no_common)"

ts_txt, txt_key_used = _load_txt_best_effort(TXT_PATH, steps_h5)
steps_txt = ts_txt.steps
print("TXT key_used:", txt_key_used)
print("TXT steps[0:5]...:", steps_txt[:5], "len=", len(steps_txt))

N = int(min(len(steps_h5), len(steps_txt)))
if N <= 1:
    raise RuntimeError("Not enough steps to compare. Check that TXT export includes multiple steps.")
steps_h5_use = [int(s) for s in steps_h5[:N]]
steps_txt_use = [int(s) for s in steps_txt[:N]]

common_steps = sorted(set(steps_h5_use).intersection(set(steps_txt_use)))
if not common_steps:
    print("(warn) No common step keys between H5 and TXT.")
    print("       Proceeding with index-based alignment: H5[i] vs TXT[i] for i in [0..N-1].")
else:
    print("(info) Found common step keys; still using index-based range [0..N-1] for fairness.")

def _region_node_ids(mr, reg_code: int) -> set[int]:
    if reg_code <= 0 or reg_code > len(mr):
        return set()
    r = mr[reg_code - 1]
    out: set[int] = set()
    for el in getattr(r, "elements", []) or []:
        out.add(int(el.node_1))
        out.add(int(el.node_2))
        out.add(int(el.node_3))
    return out

def _node_xy(mr):
    return dict(getattr(mr, "node_xy", {}) or {})

def _max_disp_over_steps(ts, node_ids: set[int], steps: list[int]) -> dict[int, float]:
    if not steps:
        return {}
    mr0 = ts.by_step[int(steps[0])]
    xy0 = _node_xy(mr0)
    out: dict[int, float] = {int(n): 0.0 for n in node_ids}
    for st in steps[1:]:
        mr = ts.by_step[int(st)]
        xy = _node_xy(mr)
        for nid in node_ids:
            p0 = xy0.get(int(nid))
            p1 = xy.get(int(nid))
            if p0 is None or p1 is None:
                continue
            try:
                d2 = (float(p1[0]) - float(p0[0])) ** 2 + (float(p1[1]) - float(p0[1])) ** 2
                d = float(d2 ** 0.5)
                if d > out[int(nid)]:
                    out[int(nid)] = d
            except Exception:
                continue
    return out

# Load step0 regions (by index)
step0_h5 = steps_h5_use[int(STEP0)]
step0_txt = steps_txt_use[int(STEP0)]
mr_h5_0 = ts_h5.by_step[int(step0_h5)]
mr_txt_0 = ts_txt.by_step[int(step0_txt)]

nodes_h5 = _region_node_ids(mr_h5_0, REG_CODE)
nodes_txt = _region_node_ids(mr_txt_0, REG_CODE)
print(f"reg_code={REG_CODE}: node_ids in H5(step={step0_h5})={len(nodes_h5)}, in TXT(step={step0_txt})={len(nodes_txt)}")
print("node_ids only in TXT:", len(nodes_txt - nodes_h5))
print("node_ids only in H5:", len(nodes_h5 - nodes_txt))

# Compare motion magnitude per node across the aligned index range
disp_h5 = _max_disp_over_steps(ts_h5, nodes_h5, steps_h5_use)
disp_txt = _max_disp_over_steps(ts_txt, nodes_txt, steps_txt_use)

# Nodes that move in TXT but are nearly static in H5
STATIC_TOL_MM = 1e-6
suspects = []
for nid in sorted(nodes_txt.intersection(nodes_h5)):
    dt = disp_txt.get(int(nid), 0.0)
    dh = disp_h5.get(int(nid), 0.0)
    if dt > 1e-3 and dh <= STATIC_TOL_MM:
        suspects.append((nid, dt, dh))

print("suspects (move in TXT, static in H5):", len(suspects))
for nid, dt, dh in suspects[:20]:
    print(f"  node {nid}: txt_max_disp={dt:.6g} mm, h5_max_disp={dh:.6g} mm")

# H5 internal check: are all reg_code nodes included in moving_node_id?
try:
    import h5py
    with h5py.File(H5_PATH, "r") as f:
        mcm = str(f.attrs.get("mesh_coords_mode", ""))
        if "by_step_moving_nodes" not in mcm.lower():
            print("(info) H5 mesh_coords_mode is not by_step_moving_nodes:", mcm)
        moving_ids = (
            set(int(x) for x in np.asarray(f["mesh/moving_node_id"][:], dtype=np.int32).tolist())
            if "mesh/moving_node_id" in f
            else set()
        )
        rc = np.asarray(f["mesh/reg_code"][:], dtype=np.int32)
        n1 = np.asarray(f["mesh/node_1"][:], dtype=np.int32)
        n2 = np.asarray(f["mesh/node_2"][:], dtype=np.int32)
        n3 = np.asarray(f["mesh/node_3"][:], dtype=np.int32)
        mask = (rc == int(REG_CODE))
        reg_nodes = set(int(x) for x in np.concatenate((n1[mask], n2[mask], n3[mask]), axis=0).tolist())
        missing_in_moving = sorted(reg_nodes - moving_ids)
        print(f"H5 moving_node_id count={len(moving_ids)}")
        print(f"H5 reg_code={REG_CODE} nodes={len(reg_nodes)}")
        print(f"reg_code nodes missing in moving_node_id={len(missing_in_moving)}")
        if missing_in_moving:
            print("  first missing node_ids:", missing_in_moving[:30])
except Exception as e:
    print("(warn) Could not inspect raw H5 moving_node_id:", e)

# Compare a couple of suspect node coordinates at first/last (by index)
step1_h5 = steps_h5_use[int(STEP1)]
step1_txt = steps_txt_use[int(STEP1)]
if suspects:
    nid = suspects[0][0]
    p_txt0 = _node_xy(ts_txt.by_step[int(step0_txt)]).get(int(nid))
    p_txt1 = _node_xy(ts_txt.by_step[int(step1_txt)]).get(int(nid))
    p_h50 = _node_xy(ts_h5.by_step[int(step0_h5)]).get(int(nid))
    p_h51 = _node_xy(ts_h5.by_step[int(step1_h5)]).get(int(nid))
    print("\nExample suspect node:")
    print(f" step0: TXT(step={step0_txt}) {p_txt0} | H5(step={step0_h5}) {p_h50}")
    print(f" step1: TXT(step={step1_txt}) {p_txt1} | H5(step={step1_h5}) {p_h51}")
else:
    print("\nNo suspects found under current thresholds. If you still see partial rotation, it may be: ")
    print("- a2 region actually includes stationary geometry in this result, or")
    print("- the TXT/H5 are not from the exact same case/result (different .mes), or")
    print("- plotting is filtering/skipping elements due to missing node coords.")

# Parametric DOE: Geometry × Electrical → Training Data Generation

**목적**: `Ratio_SlotDepth_ParallelSlot`, `Ratio_Bore` 형상 비율과 `PeakCurrent`, `PhaseAdvance` 전류 조건을 조합해
full-factorial DOE를 구성하고, 각 조합마다 Motor-CAD에서 transient FEA를 수행한 뒤 H5(메시+자기장 시계열)로 내보냅니다.

**데이터 용도**: 형상·전류 조건이 변해도 시간-도메인 자속밀도 분포(Bx, By)를 추론하는 Neural Operator 학습용

**워크플로**:
1. 기준 `.mot` 파일 열기 + 기본 파라미터 읽기
2. DOE 그리드 빌드 (geometry × electrical)
3. each DOE point → set_variable → solve → export `.mes` → process to H5
4. 전체 H5를 하나의 학습 데이터셋으로 통합

In [ ]:
# ============================================================
# 4-1) 기준 mot 파일 열기 + 기본 형상/전류 파라미터 읽기
# ============================================================
import importlib
import pathlib
import sys
import shutil
from pathlib import Path
from datetime import datetime

# Ensure repo root on path
repo_root = pathlib.Path.cwd().resolve()
while not ((repo_root / "tools").exists() or (repo_root / "tool").exists()) and repo_root != repo_root.parent:
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import tools.motorCAD.pyMCAD as _pyMCAD
import tools.motorCAD.pyMCAD.melec_req_check as _melec
importlib.reload(_pyMCAD)
importlib.reload(_melec)

from tools.motorCAD.pyMCAD.melec_req_check import get_mcad_variables, set_mcad_variables

import ansys.motorcad.core as pymotorcad

# --- 설정 ---
BASE_MOT = r"D:\KDH\Sim_4SolverX\TestCAD1.mot"
DOE_OUT_ROOT = r"D:\KDH\Sim_4SolverX\DOE_TrainingData"

# Motor-CAD 연결 (기존 인스턴스 사용)
mc = pymotorcad.MotorCAD(open_new_instance=False)
# mc.load_from_file(BASE_MOT)
mc.display_screen("E-Magnetics;FEA")

# 기본 파라미터 확인
base_params = get_mcad_variables(mc, [
    "Ratio_Bore",
    "Ratio_SlotDepth_ParallelSlot",
    "Stator_Bore",
    "Slot_Depth",
    "PeakCurrent",
    "PhaseAdvance",
    "ShaftSpeed",
    "Pole_Number",
    "Slot_Number",
])
print("=== Base Parameters ===")
for k, v in base_params.items():
    print(f"  {k}: {v}")

In [ ]:
# ============================================================
# 4-2) DOE 그리드 빌드: LHS (Geometry + Electrical 통합 샘플링)
# ============================================================
# Full-factorial 대신 Latin Hypercube Sampling으로 적은 수의 공간충전 점 생성
# 4차원: Ratio_Bore, Ratio_SlotDepth_ParallelSlot, PeakCurrent, PhaseAdvance
import importlib
import tools.pyutils.sweep as _sweep
importlib.reload(_sweep)

from tools.pyutils.sweep import DOEAxis, build_doe_lhs

# --- 기본값 기준 범위 설정 ---
rb_base = float(base_params["Ratio_Bore"])
rsd_base = float(base_params["Ratio_SlotDepth_ParallelSlot"])

all_axes = [
    # Geometry (Ratio 기반)
    DOEAxis("Ratio_Bore",
            min_val=round(rb_base * 0.90, 4),
            max_val=round(rb_base * 1.10, 4),
            steps=0),   # steps는 LHS에서 무시됨
    DOEAxis("Ratio_SlotDepth_ParallelSlot",
            min_val=round(rsd_base * 0.85, 4),
            max_val=round(rsd_base * 1.15, 4),
            steps=0),
    # Electrical
    DOEAxis("PeakCurrent",
            min_val=10.0,
            max_val=650.53,
            steps=0),
    DOEAxis("PhaseAdvance",
            min_val=0.0,
            max_val=90.0,
            steps=0),
]

# --- LHS 샘플 수 ---
# Full-factorial (3×3×6×8 = 432) 대비 크게 축소
N_SAMPLES = 40       # 학습용 데이터 포인트 수 (필요시 조절)
LHS_SEED = 42        # 재현성
LHS_CRITERION = "maximin"  # "classic" | "maximin" | "scipy"

doe_grid = build_doe_lhs(
    axes=all_axes,
    n_samples=N_SAMPLES,
    seed=LHS_SEED,
    criterion=LHS_CRITERION,
)

print(f"LHS criterion : {LHS_CRITERION}")
print(f"Dimensions    : {len(all_axes)}  ({', '.join(ax.name for ax in all_axes)})")
print(f"Total samples : {len(doe_grid)}")
print()

# Preview
for pt in doe_grid[:5]:
    print(f"  [{pt.index:3d}] geo={pt.geometry}  elec={pt.electrical}")
print("  ...")
for pt in doe_grid[-3:]:
    print(f"  [{pt.index:3d}] geo={pt.geometry}  elec={pt.electrical}")

# --- 시각화: 2D projections ---
import matplotlib.pyplot as plt
import numpy as np

fig, axes_plt = plt.subplots(1, 3, figsize=(14, 4))
geo_rb  = [pt.geometry["Ratio_Bore"] for pt in doe_grid]
geo_rsd = [pt.geometry["Ratio_SlotDepth_ParallelSlot"] for pt in doe_grid]
elec_ip = [pt.electrical["PeakCurrent"] for pt in doe_grid]
elec_ph = [pt.electrical["PhaseAdvance"] for pt in doe_grid]

axes_plt[0].scatter(geo_rb, geo_rsd, c=elec_ip, cmap="viridis", edgecolors="k", s=40)
axes_plt[0].set_xlabel("Ratio_Bore"); axes_plt[0].set_ylabel("Ratio_SlotDepth")
axes_plt[0].set_title("Geometry (color=Ipk)")

axes_plt[1].scatter(elec_ip, elec_ph, c=geo_rb, cmap="plasma", edgecolors="k", s=40)
axes_plt[1].set_xlabel("PeakCurrent [A]"); axes_plt[1].set_ylabel("PhaseAdvance [°]")
axes_plt[1].set_title("Electrical (color=Ratio_Bore)")

axes_plt[2].scatter(geo_rb, elec_ip, c=elec_ph, cmap="coolwarm", edgecolors="k", s=40)
axes_plt[2].set_xlabel("Ratio_Bore"); axes_plt[2].set_ylabel("PeakCurrent [A]")
axes_plt[2].set_title("Cross (color=PhaseAdv)")

fig.suptitle(f"LHS DOE ({N_SAMPLES} points, {LHS_CRITERION})", fontweight="bold")
fig.tight_layout()
plt.show()

# 4-3) DOE Batch Solve + Export H5

**모듈화된 파이프라인** (`tools.motorCAD.pyMCAD.doe_batch`)을 사용합니다.

| 단계 | 함수 | 설명 |
|------|------|------|
| Solve | `doe_solve_single()` | .mot 로드 → 변수 설정 → solve → **.mes 보존** + `doe_condition.json` 저장 |
| Export | `doe_export_h5()` | 보존된 .mes → H5/TXT 변환 |
| Batch | `doe_batch_run()` | 위 두 단계를 전체 DOE grid에 대해 순회 |

**`phases` 옵션**으로 단계를 분리할 수 있습니다:
- `phases=["solve", "export"]` — solve + export 한꺼번에 (기본)
- `phases=["solve"]` — solve만 (.mes 보존, 나중에 export)
- 나중에 `doe_export_from_saved()` 로 .mes → H5 재변환 가능

**디렉토리 구조** (case별):
```
case_NNNN/
├── doe_condition.json   ← {geometry, electrical, readback, base_mot, timestamp}
├── mes/                 ← .mes 원본 (solve 직후 복사)
│   ├── OnLoadTorque_result_1.mes
│   └── StaticLoad_result_1.mes
└── postproc/            ← H5/TXT (export 단계)
    └── Mag_OnLoadTorque_result_1.h5
```

In [ ]:
# ============================================================
# 4-3) DOE Batch Solve → Export H5  (modularized pipeline)
# ============================================================
import importlib
from pathlib import Path

import tools.motorCAD.pyMCAD.doe_batch as _doe_batch
importlib.reload(_doe_batch)

from tools.motorCAD.pyMCAD.doe_batch import doe_batch_run

# --- 설정 ---
DOE_OUT = Path(DOE_OUT_ROOT)

# phases 옵션:
#   ["solve", "export"]  — solve + .mes 보존 + H5 export (기본)
#   ["solve"]            — solve + .mes 보존만 (나중에 export)
PHASES = ["solve", "export"]

FIRST_STEP = 1
FINAL_STEP = 45
MAG_H5_MESH_COORDS = "by_step_moving_nodes"
MAG_COLUMNS = "RegCode,Bx,By,A,J,Je"   # Je 추가 시: "RegCode,Bx,By,A,J,Je"
PLOT_MODE = "none"  # 배치 모드 → 그래프 저장 안 함 (속도 우선)

# doe_grid, BASE_MOT 는 이전 셀에서 정의됨
manifest = doe_batch_run(
    mc,
    doe_grid,
    base_mot=BASE_MOT,
    doe_out_root=DOE_OUT,
    phases=PHASES,
    first_step=FIRST_STEP,
    final_step=FINAL_STEP,
    mag_columns=MAG_COLUMNS,
    mag_h5_mesh_coords=MAG_H5_MESH_COORDS,
    plot_mode=PLOT_MODE,
)

# 4-3b) 저장된 .mes로부터 H5 재변환 (Re-export)

이미 solve 완료된 `case_NNNN/mes/` 폴더의 .mes 파일을 사용하여 **solve 없이** H5를 재생성합니다.

**용도:**
- 컬럼 추가 (예: `Je` 추가) 시 re-solve 없이 H5만 재변환
- H5 mesh coords 옵션 변경 (`by_step` ↔ `by_step_moving_nodes`)
- 일부 case만 선택적으로 재변환 (`case_indices` 지정)

In [ ]:
# ============================================================
# 4-3b) Re-export H5 from saved .mes  (no re-solve)
# ============================================================
import importlib
from pathlib import Path

import tools.motorCAD.pyMCAD.doe_batch as _doe_batch
importlib.reload(_doe_batch)

from tools.motorCAD.pyMCAD.doe_batch import doe_export_from_saved

DOE_OUT = Path(DOE_OUT_ROOT)

# --- 설정 ---
# case_indices=None → 전체 case 재변환
# case_indices=[0, 5, 10] → 특정 case만 재변환
CASE_INDICES = None

# 컬럼 변경 예시: Je 추가
MAG_COLUMNS = "RegCode,Bx,By,A,J"   # "RegCode,Bx,By,A,J,Je" 로 변경 가능

result = doe_export_from_saved(
    mc,
    doe_out_root=DOE_OUT,
    case_indices=CASE_INDICES,
    first_step=1,
    final_step=45,
    mag_columns=MAG_COLUMNS,
    mag_h5_mesh_coords="by_step_moving_nodes",
    plot_mode="none",
)

# 4-4) Docker로 학습 데이터 전송 + Training 실행

생성된 H5 파일과 manifest를 Docker 컨테이너(`friendly_knuth`)의 `/workspace/doe_data/`로 복사한 뒤,
PhysicsNeMo **MeshGraphNet**으로 multi-case(형상+전류 조건 포함) 학습을 수행합니다.

In [ ]:
# ============================================================
# 4-4a) Docker 컨테이너로 DOE H5 + manifest 전송
# ============================================================
import subprocess, json
from pathlib import Path

CONTAINER = "friendly_knuth"
DOCKER_DATA_DIR = "/workspace/doe_data"

# 1) 컨테이너에 디렉토리 생성
subprocess.run(["docker", "exec", CONTAINER, "mkdir", "-p", DOCKER_DATA_DIR], check=True)

# 2) manifest 전송
manifest_local = Path(DOE_OUT_ROOT) / "doe_manifest.json"
subprocess.run(["docker", "cp", str(manifest_local), f"{CONTAINER}:{DOCKER_DATA_DIR}/doe_manifest.json"], check=True)
print(f"✓ manifest → {DOCKER_DATA_DIR}/doe_manifest.json")

# 3) 각 case의 H5 파일 전송 (tree 구조 유지)
with open(manifest_local) as f:
    mf = json.load(f)

h5_count = 0
for case in mf["cases"]:
    if not case.get("h5_paths"):
        continue
    idx = case["index"]
    remote_case = f"{DOCKER_DATA_DIR}/case_{idx:04d}/postproc"
    subprocess.run(["docker", "exec", CONTAINER, "mkdir", "-p", remote_case], check=True)

    for h5p in case["h5_paths"]:
        h5_local = Path(h5p)
        if h5_local.exists():
            subprocess.run(["docker", "cp", str(h5_local), f"{CONTAINER}:{remote_case}/{h5_local.name}"], check=True)
            h5_count += 1

    # meta.json도 같이
    meta_local = Path(DOE_OUT_ROOT) / f"case_{idx:04d}" / "meta.json"
    if meta_local.exists():
        subprocess.run(["docker", "cp", str(meta_local),
                        f"{CONTAINER}:{DOCKER_DATA_DIR}/case_{idx:04d}/meta.json"], check=True)

print(f"✓ {h5_count} H5 files transferred to {CONTAINER}:{DOCKER_DATA_DIR}")

# 4) 확인
result = subprocess.run(
    ["docker", "exec", CONTAINER, "find", DOCKER_DATA_DIR, "-name", "*.h5", "-type", "f"],
    capture_output=True, text=True
)
h5_in_docker = [l for l in result.stdout.strip().split("\n") if l.strip()]
print(f"  H5 in Docker: {len(h5_in_docker)}")
for p in h5_in_docker[:5]:
    print(f"    {p}")
if len(h5_in_docker) > 5:
    print(f"    ... ({len(h5_in_docker) - 5} more)")

In [ ]:
# ============================================================
# 4-4b) Training script를 Docker로 복사 + 학습 실행
# ============================================================
import subprocess

CONTAINER = "friendly_knuth"
TRAIN_SCRIPT = r"D:\KDH\NvidiaNemo\train_doe_meshgraphnet.py"

# 1) 학습 스크립트 전송
subprocess.run(["docker", "cp", TRAIN_SCRIPT, f"{CONTAINER}:/workspace/train_doe_meshgraphnet.py"], check=True)
print("✓ train_doe_meshgraphnet.py → /workspace/")

# 2) 학습 실행 (GPU, RTX 3090)
#    processor_size=15 : message-passing 레이어 15개 (모터 메시 복잡도에 적합)
#    hidden_dim=128    : 128 hidden units
#    epochs=60         : 60 에폭
#    batch_size=4      : RTX 3090 24GB 기준 안전한 배치
#    node feature 11차원: [x,y,A,J,region,t,rot, Ratio_Bore, Ratio_SlotDepth, Ipk, Phase]
#
# 백그라운드 실행 + 로그 파일 저장
cmd_str = (
    "nohup python /workspace/train_doe_meshgraphnet.py "
    "--data-dir /workspace/doe_data "
    "--epochs 60 "
    "--batch-size 4 "
    "--lr 1e-3 "
    "--processor-size 15 "
    "--hidden-dim 128 "
    "--train-ratio 0.8 "
    "--ckpt /workspace/doe_meshgraphnet_ckpt.pt "
    "> /workspace/train_doe.log 2>&1 &"
)

result = subprocess.run(
    ["docker", "exec", CONTAINER, "bash", "-c", cmd_str],
    capture_output=True, text=True,
)
print(f"✓ Training launched in background (returncode={result.returncode})")
print()
print("  로그 실시간 확인:")
print("    docker exec friendly_knuth tail -f /workspace/train_doe.log")
print()
print("  체크포인트 위치: /workspace/doe_meshgraphnet_ckpt.pt")

In [ ]:
# ============================================================
# 4-4c) 학습 진행 확인 + 체크포인트 가져오기
# ============================================================
import subprocess

CONTAINER = "friendly_knuth"

# 1) 학습 프로세스 상태 확인
ps = subprocess.run(
    ["docker", "exec", CONTAINER, "bash", "-c",
     "ps aux | grep train_doe_meshgraphnet | grep -v grep"],
    capture_output=True, text=True
)
if ps.stdout.strip():
    print("▶ Training RUNNING:")
    print(ps.stdout.strip())
else:
    print("Training process not running (finished or not started)")

# 2) 최신 출력 확인 (프로세스의 stdout)
log = subprocess.run(
    ["docker", "exec", CONTAINER, "bash", "-c",
     "cat /tmp/train_doe.log 2>/dev/null || echo '(no log file found)'"],
    capture_output=True, text=True
)
# 대안: foreground로 실행했으면 docker logs 사용
print("\n--- Recent log ---")
print(log.stdout[-2000:] if len(log.stdout) > 2000 else log.stdout)

# 3) 체크포인트 확인
ckpt = subprocess.run(
    ["docker", "exec", CONTAINER, "bash", "-c",
     "ls -lh /workspace/doe_meshgraphnet_ckpt.pt 2>/dev/null || echo 'checkpoint not found yet'"],
    capture_output=True, text=True
)
print("\n--- Checkpoint ---")
print(ckpt.stdout.strip())

# 4) 체크포인트 로컬로 가져오기 (학습 완료 후 실행)
FETCH_CKPT = False   # True로 바꾸면 로컬로 복사
if FETCH_CKPT:
    local_ckpt = Path(DOE_OUT_ROOT) / "doe_meshgraphnet_ckpt.pt"
    subprocess.run(["docker", "cp",
                    f"{CONTAINER}:/workspace/doe_meshgraphnet_ckpt.pt",
                    str(local_ckpt)], check=True)
    print(f"✓ Checkpoint → {local_ckpt}")

# 모터 와류 Time-stepping FEA에 적합한 Neural Operator 모델 비교

## 문제 특성
- **비정형 메시** (삼각형 FEM), 공극에 **sliding band** → 고정자 고정 + 회전자 회전
- **시계열** (time-stepping): 와류(eddy current) 포함, 과도 해석
- **파라메트릭**: 형상(Ratio_Bore, Ratio_SlotDepth) + 전류(Ipk, PhaseAdvance)가 변함
- **출력**: 메시 전체의 (Bx, By) 자속밀도 분포

## 적합 모델 비교

| 모델 | 적합도 | 장점 | 단점 |
|------|-------|------|------|
| **MeshGraphNet** ⭐ | ★★★★★ | 비정형 메시 네이티브, 메시 토폴로지가 변해도 학습 가능, sliding band edge 처리 가능 | autoregressive rollout 시 오차 누적 |
| **Transolver** | ★★★★☆ | Transformer 기반 PDE solver, 비정형 메시 지원, condition feature 주입 용이 | 메시가 클 때 attention 메모리 큼 |
| **FIGConvUNet** | ★★★★☆ | Field-Informed Graph Conv + U-Net, 멀티스케일 | 아직 motor 도메인 검증 부족 |
| **DPOT** | ★★★☆☆ | Differential PDE Operator Transformer, 시계열 지원 | 정형 격자 기반 설계, 비정형 메시에 interpolation 필요 |
| **FNO** | ★★☆☆☆ | 빠른 학습, Fourier space 연산 | 정형 격자 전용 → 비정형 모터 메시에 부적합 |
| **DeepONet** | ★★★☆☆ | operator learning, 파라미터 입력 자연스럽게 branch net에 주입 | trunk net이 좌표 기반 → 메시 변화에 약함 |

## 추천 전략

### 1단계: MeshGraphNet (현재 구현)
- 각 시간 스텝을 독립 그래프로 학습 (condition feature로 형상+전류 주입)
- `node_feature = [x, y, A, J, region, t, rot, Ratio_Bore, Ratio_SlotDepth, Ipk, Phase]`
- 장점: 메시 토폴로지 변화(형상 변경)에도 message-passing으로 자연스럽게 대응

### 2단계: Autoregressive MeshGraphNet (rollout)
- 이전 스텝 출력(Bx,By)을 다음 스텝 입력에 포함 → 시계열 외삽
- sliding band 경계 edge를 시간 스텝마다 재구성 (회전자 위치 반영)

### 3단계: 하이브리드 고려
- **BiStrideMeshGraphNet**: 멀티스케일 message-passing (stator coarse + air-gap fine)
- **Transolver**: 메시가 큰 경우 attention window로 확장

### Sliding Band 처리 팁
- 고정자/회전자 메시는 별도로 static, 공극 sliding band 경계의 edge만 시간 스텝마다 갱신
- `reg_code`로 stator/rotor/airgap 노드를 구분 → GNN edge 구성 시 활용
- `moving_node_indices`로 회전 노드만 좌표 업데이트 (H5의 by_step_moving_nodes 모드)

# 5) 추론 & 시각화: MeshGraphNet vs FEA 비교

학습 완료된 체크포인트를 Docker에서 실행하여 FEA ground truth 대비 예측값을 비교합니다.
- **Field Comparison**: Bx, By 분포 (FEA / Prediction / Error)
- **Scatter Plot**: 예측값 vs 실측값 (R² score)
- **Error Histogram**: 노드별 오차 분포
- **Global Metrics**: 전체 DOE 케이스 통합 RMSE, NRMSE%

In [ ]:
# ============================================================
# 5-1) 추론 스크립트를 Docker에 복사 & 실행
# ============================================================
import subprocess, json
from pathlib import Path

CONTAINER = "friendly_knuth"
INFER_SCRIPT = r"D:\KDH\NvidiaNemo\infer_doe_meshgraphnet.py"
DOCKER_DATA_DIR = "/workspace/doe_data"
CKPT_PATH = "/workspace/doe_meshgraphnet_ckpt.pt"
PLOT_DIR = "/workspace/plots"

# 1) 추론 스크립트 복사
subprocess.run(["docker", "cp", INFER_SCRIPT,
                f"{CONTAINER}:/workspace/infer_doe_meshgraphnet.py"], check=True)
print("✓ infer_doe_meshgraphnet.py → Docker")

# 2) 추론 실행 (중간 케이스 + 전체 메트릭)
cmd = (
    f"cd /workspace && python -u infer_doe_meshgraphnet.py "
    f"--ckpt {CKPT_PATH} --data-dir {DOCKER_DATA_DIR} "
    f"--save-dir {PLOT_DIR} --all-metrics 2>&1"
)
result = subprocess.run(
    ["docker", "exec", CONTAINER, "bash", "-c", cmd],
    capture_output=True, text=True, timeout=600
)
print(result.stdout[-5000:] if len(result.stdout) > 5000 else result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr[-2000:])

In [ ]:
# ============================================================
# 5-2) Docker에서 플롯 이미지 가져와 표시
# ============================================================
import subprocess, glob
from pathlib import Path
from IPython.display import display, Image as IPImage

CONTAINER = "friendly_knuth"
LOCAL_PLOT_DIR = Path(DOE_OUT_ROOT) / "plots"
LOCAL_PLOT_DIR.mkdir(parents=True, exist_ok=True)

# Docker → Local: 모든 plot PNG 복사
subprocess.run(
    ["docker", "cp", f"{CONTAINER}:/workspace/plots/.", str(LOCAL_PLOT_DIR)],
    check=True
)

# 플롯 파일 목록
plot_files = sorted(LOCAL_PLOT_DIR.glob("*.png"))
print(f"📊 {len(plot_files)} plots fetched to {LOCAL_PLOT_DIR}\n")

# 학습 이력 먼저 표시
for pf in plot_files:
    if "training_history" in pf.name:
        print(f"=== {pf.name} ===")
        display(IPImage(filename=str(pf), width=700))
        break

# Field comparison, scatter, histogram 순서로 표시
for suffix in ["field_comparison", "scatter", "error_hist", "all_cases_scatter"]:
    for pf in plot_files:
        if suffix in pf.name:
            print(f"\n=== {pf.name} ===")
            display(IPImage(filename=str(pf), width=800 if "field" in suffix else 700))

In [ ]:
# ============================================================
# 5-3) 체크포인트 로컬 로드 + 학습 이력 시각화 (CPU)
# ============================================================
import torch
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

CKPT_LOCAL = Path(r"D:\KDH\NvidiaNemo\doe_meshgraphnet_ckpt.pt")

ckpt = torch.load(CKPT_LOCAL, map_location="cpu", weights_only=False)

print(f"Epoch: {ckpt['epoch']}")
print(f"Best val MSE: {ckpt['val_hist'][-1]:.6f}" if 'val_hist' in ckpt else "")
print(f"Args: {ckpt.get('args', {})}")

# --- Training history ---
tr = ckpt.get("train_hist", [])
va = ckpt.get("val_hist", [])

fig, ax = plt.subplots(figsize=(9, 5))
epochs = list(range(1, len(tr) + 1))
ax.semilogy(epochs, tr, "b-", label=f"Train MSE (final={tr[-1]:.6f})", linewidth=1.5)
ax.semilogy(epochs, va, "r-", label=f"Val MSE (final={va[-1]:.6f})", linewidth=1.5)
ax.set_xlabel("Epoch", fontsize=12)
ax.set_ylabel("MSE Loss (log)", fontsize=12)
ax.set_title("MeshGraphNet DOE Training Convergence\n"
             f"40 LHS cases × 45 timesteps = 1800 graphs, 2.33M params", fontsize=12)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
fig.tight_layout()
plt.show()

# --- Normalisation stats summary ---
for name in ["x_mean", "x_std", "y_mean", "y_std", "e_mean", "e_std"]:
    t = ckpt[name]
    print(f"  {name}: shape={list(t.shape)}, values={t.squeeze().tolist()}"
          if t.numel() <= 20 else f"  {name}: shape={list(t.shape)}")

In [ ]:
# ============================================================
# 5-4) 다중 케이스 / 다중 시간 스텝 추론 (Docker 실행)
#      — 특정 케이스 여러 시간 스텝의 Bx/By 예측 비교
# ============================================================
import subprocess

CONTAINER = "friendly_knuth"

# 여러 케이스에 대해 추론 실행 (case 0, 10, 20, 30)
for cidx in [0, 10, 20, 30]:
    cmd = (
        f"cd /workspace && python -u infer_doe_meshgraphnet.py "
        f"--ckpt /workspace/doe_meshgraphnet_ckpt.pt "
        f"--data-dir /workspace/doe_data "
        f"--case-idx {cidx} --step-idx 22 "
        f"--save-dir /workspace/plots 2>&1 | tail -15"
    )
    result = subprocess.run(
        ["docker", "exec", CONTAINER, "bash", "-c", cmd],
        capture_output=True, text=True, timeout=300
    )
    print(f"\n{'='*60}")
    print(f"Case {cidx}:")
    print(result.stdout)

# 결과 이미지 로컬로 복사
LOCAL_PLOT_DIR = Path(DOE_OUT_ROOT) / "plots"
subprocess.run(
    ["docker", "cp", f"{CONTAINER}:/workspace/plots/.", str(LOCAL_PLOT_DIR)],
    check=True
)
print(f"\n✓ All plots → {LOCAL_PLOT_DIR}")
print(f"  Files: {len(list(LOCAL_PLOT_DIR.glob('*.png')))}")

In [ ]:
# ============================================================
# 5-5) 다중 케이스 결과 비교 갤러리
# ============================================================
from pathlib import Path
from IPython.display import display, Image as IPImage

LOCAL_PLOT_DIR = Path(DOE_OUT_ROOT) / "plots"
plot_files = sorted(LOCAL_PLOT_DIR.glob("*.png"))

# 케이스별 field comparison 표시
field_plots = [p for p in plot_files if "field_comparison" in p.name]
scatter_plots = [p for p in plot_files if "scatter" in p.name and "all_cases" not in p.name]

print(f"총 {len(plot_files)} 플롯 파일\n")

for pf in field_plots:
    print(f"{'─'*50}")
    print(f"📊 {pf.name}")
    display(IPImage(filename=str(pf), width=900))

print(f"\n{'='*60}\n📈 Scatter Plots (Predicted vs FEA):\n")
for pf in scatter_plots:
    print(f"📊 {pf.name}")
    display(IPImage(filename=str(pf), width=700))

# All-cases combined scatter
all_scatter = [p for p in plot_files if "all_cases_scatter" in p.name]
if all_scatter:
    print(f"\n{'='*60}\n🌐 전체 케이스 통합 Scatter:")
    display(IPImage(filename=str(all_scatter[0]), width=700))

# 6) Interactive FEA vs MeshGraphNet 비교 GUI

슬라이더로 **DOE 케이스**와 **시간 스텝**을 바꿔가며 실시간 비교:  
- **좌**: FEA Ground Truth  |  **중**: MeshGraphNet 예측  |  **우**: Error (Pred − FEA)  
- Quantity 선택: Bx, By, |B|  
- 노드별 RMSE, R², NRMSE 실시간 표시

> 사전 조건: Docker에서 `export_predictions_npz.py` 실행 완료 → 로컬에 NPZ 복사

In [ ]:
# ============================================================
# 6-1) Docker에서 NPZ 예측 파일 전송 + 로컬 로드
# ============================================================
import subprocess
from pathlib import Path

CONTAINER = "friendly_knuth"
DOE_OUT_ROOT = globals().get("DOE_OUT_ROOT", r"D:\KDH\Sim_4SolverX\DOE_TrainingData")
NPZ_LOCAL = Path(DOE_OUT_ROOT) / "pred_npz"
NPZ_LOCAL.mkdir(parents=True, exist_ok=True)

# Docker에서 NPZ 가져오기
result = subprocess.run(
    ["docker", "cp", f"{CONTAINER}:/workspace/pred_npz/.", str(NPZ_LOCAL)],
    capture_output=True, text=True
)
if result.returncode == 0:
    print(f"✓ NPZ files → {NPZ_LOCAL}")
else:
    print("Docker cp failed (maybe still running?):", result.stderr[:200])

npz_files = sorted(NPZ_LOCAL.glob("case_*_predictions.npz"))
print(f"  {len(npz_files)} NPZ files found")
for f in npz_files[:5]:
    print(f"    {f.name}  ({f.stat().st_size/1024:.0f} KB)")
if len(npz_files) > 5:
    print(f"    ... and {len(npz_files)-5} more")

In [ ]:
# ============================================================
# 6-2) Interactive 
# FEA vs MeshGraphNet Comparison GUI (ipywidgets)
# ============================================================
#
# 기존 interactive_magnetic_plot 의 SelectionSlider + Output 패턴을 그대로 따르되
# 좌/중/우 3-panel (FEA | Prediction | Error) 으로 확장합니다.

import json
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.tri import Triangulation
from pathlib import Path

try:
    import ipywidgets as widgets
    from IPython.display import display
except ImportError:
    widgets = None
    print("[WARN] ipywidgets not available. Install: pip install ipywidgets")

# ---- 1) Load all NPZ into a dict keyed by case_index ----
NPZ_LOCAL = Path(globals().get("DOE_OUT_ROOT", r"D:\KDH\Sim_4SolverX\DOE_TrainingData")) / "pred_npz"

_CASE_DATA = {}
for npz_path in sorted(NPZ_LOCAL.glob("case_*_predictions.npz")):
    d = np.load(npz_path, allow_pickle=True)
    cidx = int(d["case_index"])
    _CASE_DATA[cidx] = {
        "pos":      d["pos"],        # (n_steps, n_nodes, 2)
        "true_bxy": d["true_bxy"],   # (n_steps, n_nodes, 2)
        "pred_bxy": d["pred_bxy"],   # (n_steps, n_nodes, 2)
        "steps":    d["steps"],      # (n_steps,)
        "times":    d["times"],      # (n_steps,)
        "condition": json.loads(str(d["condition"])),
        "path": npz_path.name,
    }
print(f"Loaded {len(_CASE_DATA)} case predictions")

if not _CASE_DATA or widgets is None:
    raise RuntimeError("No prediction data or ipywidgets not available")

# ---- 2) Build widgets ----
case_indices = sorted(_CASE_DATA.keys())

# Case dropdown with condition info
case_options = []
for ci in case_indices:
    cond = _CASE_DATA[ci]["condition"]
    label = (f"Case {ci:04d}  RB={cond.get('Ratio_Bore',0):.3f}  "
             f"RSD={cond.get('Ratio_SlotDepth_ParallelSlot',0):.3f}  "
             f"Ipk={cond.get('PeakCurrent',0):.0f}A  "
             f"Ph={cond.get('PhaseAdvance',0):.0f}°")
    case_options.append((label, ci))

case_dd = widgets.Dropdown(
    options=case_options,
    value=case_indices[len(case_indices)//2],  # middle case default
    description="Case",
    style={"description_width": "50px"},
    layout=widgets.Layout(width="600px"),
)

# Step slider (updated dynamically when case changes)
_init_steps = list(_CASE_DATA[case_dd.value]["steps"])
step_slider = widgets.SelectionSlider(
    options=_init_steps,
    value=_init_steps[len(_init_steps)//2],
    description="Step",
    continuous_update=False,
    layout=widgets.Layout(width="650px"),
)

# Quantity dropdown
qty_dd = widgets.Dropdown(
    options=[("Bx", "bx"), ("By", "by"), ("|B|", "bmag"),
             ("Error Bx", "err_bx"), ("Error By", "err_by"), ("Error |B|", "err_bmag")],
    value="bmag",
    description="Qty",
    style={"description_width": "40px"},
)

# Point size
size_slider = widgets.FloatSlider(
    value=2.0, min=0.2, max=20.0, step=0.2,
    description="size",
    continuous_update=False,
    readout_format=".1f",
    layout=widgets.Layout(width="250px"),
)

# Layout mode
layout_dd = widgets.Dropdown(
    options=[("3-panel: FEA | Pred | Error", "3panel"),
             ("Single: selected qty only", "single")],
    value="3panel",
    description="Layout",
    style={"description_width": "55px"},
)

# Metrics label
metrics_label = widgets.HTML(value="<i>Select a case and step</i>")

out = widgets.Output()
_last_fig = {"fig": None}


# ---- 3) Extract field arrays for one (case, step) ----
def _get_fields(case_idx, step_val):
    cd = _CASE_DATA[case_idx]
    si = int(np.searchsorted(cd["steps"], step_val))
    si = min(si, len(cd["steps"]) - 1)
    pos = cd["pos"][si]          # (n, 2)
    true_bxy = cd["true_bxy"][si]  # (n, 2)
    pred_bxy = cd["pred_bxy"][si]  # (n, 2)
    t = cd["times"][si]
    return pos, true_bxy, pred_bxy, float(t)


def _compute_field(true_bxy, pred_bxy, qty):
    if qty == "bx":
        return true_bxy[:, 0], pred_bxy[:, 0], "Bx [T]"
    elif qty == "by":
        return true_bxy[:, 1], pred_bxy[:, 1], "By [T]"
    elif qty == "bmag":
        bt = np.sqrt(true_bxy[:, 0]**2 + true_bxy[:, 1]**2)
        bp = np.sqrt(pred_bxy[:, 0]**2 + pred_bxy[:, 1]**2)
        return bt, bp, "|B| [T]"
    elif qty == "err_bx":
        return true_bxy[:, 0], pred_bxy[:, 0], "Bx Error [T]"
    elif qty == "err_by":
        return true_bxy[:, 1], pred_bxy[:, 1], "By Error [T]"
    elif qty == "err_bmag":
        bt = np.sqrt(true_bxy[:, 0]**2 + true_bxy[:, 1]**2)
        bp = np.sqrt(pred_bxy[:, 0]**2 + pred_bxy[:, 1]**2)
        return bt, bp, "|B| Error [T]"
    return true_bxy[:, 0], pred_bxy[:, 0], "?"


def _metrics_html(true_v, pred_v, label):
    err = pred_v - true_v
    mae = np.abs(err).mean()
    rmse = np.sqrt((err**2).mean())
    r2 = 1.0 - np.sum(err**2) / (np.sum((true_v - true_v.mean())**2) + 1e-12)
    rng = true_v.max() - true_v.min()
    nrmse = rmse / (rng + 1e-12) * 100
    return (f"<b>{label}</b>  &nbsp; MAE={mae:.4f}T &nbsp; RMSE={rmse:.4f}T &nbsp; "
            f"R²={r2:.5f} &nbsp; NRMSE={nrmse:.2f}% &nbsp; MaxErr={np.abs(err).max():.4f}T")


# ---- 4) Draw callback ----
def _draw(*_):
    with out:
        out.clear_output(wait=True)
        try:
            if _last_fig["fig"] is not None:
                plt.close(_last_fig["fig"])
        except Exception:
            pass

        ci = case_dd.value
        step_val = step_slider.value
        pos, true_bxy, pred_bxy, t_s = _get_fields(ci, step_val)
        qty = qty_dd.value
        sz = size_slider.value
        x, y = pos[:, 0], pos[:, 1]

        true_v, pred_v, label = _compute_field(true_bxy, pred_bxy, qty)
        err_v = pred_v - true_v

        mode = layout_dd.value

        if mode == "3panel":
            fig, axes = plt.subplots(1, 3, figsize=(18, 6))

            # Shared color range for FEA & Pred
            vmin = min(true_v.min(), pred_v.min())
            vmax = max(true_v.max(), pred_v.max())

            for ax, vals, title in zip(
                axes[:2],
                [true_v, pred_v],
                [f"FEA: {label}", f"MeshGraphNet: {label}"],
            ):
                sc = ax.scatter(x, y, c=vals, s=sz, cmap="jet", vmin=vmin, vmax=vmax,
                                edgecolors="none", rasterized=True)
                plt.colorbar(sc, ax=ax, fraction=0.046, pad=0.04)
                ax.set_aspect("equal")
                ax.set_title(title, fontsize=10)
                ax.set_xlabel("x [mm]")
                ax.set_ylabel("y [mm]")

            # Error panel with diverging colormap
            vlim = max(abs(err_v.min()), abs(err_v.max()), 1e-6)
            sc_err = axes[2].scatter(x, y, c=err_v, s=sz, cmap="RdBu_r",
                                     vmin=-vlim, vmax=vlim,
                                     edgecolors="none", rasterized=True)
            plt.colorbar(sc_err, ax=axes[2], fraction=0.046, pad=0.04)
            axes[2].set_aspect("equal")
            axes[2].set_title(f"Error (Pred−FEA): {label}", fontsize=10)
            axes[2].set_xlabel("x [mm]")
            axes[2].set_ylabel("y [mm]")

            cond = _CASE_DATA[ci]["condition"]
            fig.suptitle(
                f"Case {ci:04d} | Step {step_val} | t={t_s*1e3:.3f}ms | "
                f"RB={cond.get('Ratio_Bore',0):.3f} RSD={cond.get('Ratio_SlotDepth_ParallelSlot',0):.3f} "
                f"Ipk={cond.get('PeakCurrent',0):.0f}A Ph={cond.get('PhaseAdvance',0):.0f}°",
                fontsize=11,
            )
        else:
            # Single panel: show selected qty only
            fig, ax = plt.subplots(figsize=(8, 6))
            if "err" in qty:
                vlim = max(abs(err_v.min()), abs(err_v.max()), 1e-6)
                sc = ax.scatter(x, y, c=err_v, s=sz, cmap="RdBu_r", vmin=-vlim, vmax=vlim,
                                edgecolors="none", rasterized=True)
            else:
                sc = ax.scatter(x, y, c=pred_v, s=sz, cmap="jet", edgecolors="none", rasterized=True)
            plt.colorbar(sc, ax=ax, fraction=0.046, pad=0.04)
            ax.set_aspect("equal")
            ax.set_title(f"Case {ci:04d} Step {step_val}: {label}")
            ax.set_xlabel("x [mm]")
            ax.set_ylabel("y [mm]")

        fig.tight_layout(rect=[0, 0, 1, 0.95] if mode == "3panel" else None)
        _last_fig["fig"] = fig
        plt.show()

        backend = str(matplotlib.get_backend()).lower()
        if "inline" in backend or backend.endswith("agg") or "agg" in backend:
            plt.close(fig)

    # Update metrics
    metrics_label.value = _metrics_html(true_v, pred_v, label)


# ---- 5) Sync step slider when case changes ----
def _on_case_change(*_):
    ci = case_dd.value
    new_steps = list(_CASE_DATA[ci]["steps"])
    step_slider.options = new_steps
    step_slider.value = new_steps[len(new_steps)//2]

case_dd.observe(_on_case_change, names="value")

# ---- 6) Wire up draw ----
step_slider.observe(_draw, names="value")
qty_dd.observe(_draw, names="value")
size_slider.observe(_draw, names="value")
layout_dd.observe(_draw, names="value")

# ---- 7) Layout & display ----
display(widgets.VBox([
    widgets.HBox([case_dd]),
    widgets.HBox([step_slider]),
    widgets.HBox([qty_dd, layout_dd, size_slider]),
    metrics_label,
    out,
]))
_draw()

# 7) Merantix MSO — MGN vs FEM Comparison (multiscale-pde-operators)

Merantix `multiscale-pde-operators` 논문의 **MGN (EncoderProcessorDecoder)** 모델을 
DOE 데이터(40 LHS cases × 45 timesteps = 1,800 graphs)로 학습한 결과입니다.

- **모델**: EncoderProcessorDecoder, 721K params, 15 message-passing layers
- **학습**: 200 epochs, cosine annealing LR, batch=2, accum=8
- **테스트**: 180 samples (1,234,308 nodes)
- **Docker**: `friendly_knuth`, RTX 3090, PhysicsNeMo 26.03

In [ ]:
# ============================================================
# 7-1) MGN vs FEM 비교 결과 — Summary + Training History
# ============================================================
import json
from pathlib import Path
from IPython.display import display, Image as IPImage, HTML

COMPARE_DIR = Path(r"D:\KDH\NvidiaNemo\mgn_fem_comparison")

# Load summary
with open(COMPARE_DIR / "test_summary.json") as f:
    summary = json.load(f)

tm = summary["total_metrics"]
ps = summary["per_sample_summary"]
tr = summary["training"]

html = f"""
<h3>MGN (EncoderProcessorDecoder) — Test Set Results</h3>
<table style="border-collapse:collapse; font-size:14px;">
<tr style="background:#f0f0f0"><th style="padding:6px 12px; text-align:left">Metric</th><th style="padding:6px 12px">Value</th></tr>
<tr><td style="padding:4px 12px">Model</td><td style="padding:4px 12px"><b>{summary['model']}</b></td></tr>
<tr><td style="padding:4px 12px">Parameters</td><td style="padding:4px 12px">{summary['params']:,}</td></tr>
<tr><td style="padding:4px 12px">Test Samples</td><td style="padding:4px 12px">{summary['test_samples']} ({summary['total_nodes']:,} nodes)</td></tr>
<tr style="background:#fff3cd"><td style="padding:4px 12px"><b>RMSE</b></td><td style="padding:4px 12px"><b>{tm['RMSE']:.4f}</b></td></tr>
<tr style="background:#fff3cd"><td style="padding:4px 12px"><b>nRMSE</b></td><td style="padding:4px 12px"><b>{tm['nRMSE_pct']:.2f}%</b></td></tr>
<tr style="background:#d4edda"><td style="padding:4px 12px"><b>R²</b></td><td style="padding:4px 12px"><b>{tm['R2']:.4f}</b></td></tr>
<tr><td style="padding:4px 12px">MAE</td><td style="padding:4px 12px">{tm['MAE']:.4f}</td></tr>
<tr><td style="padding:4px 12px">MaxErr</td><td style="padding:4px 12px">{tm['MaxErr']:.4f}</td></tr>
<tr><td style="padding:4px 12px">Per-sample R² (median)</td><td style="padding:4px 12px">{ps['r2_median']:.4f}</td></tr>
<tr><td style="padding:4px 12px">Per-sample nRMSE (median)</td><td style="padding:4px 12px">{ps['nrmse_median_pct']:.1f}%</td></tr>
<tr style="background:#f0f0f0"><td style="padding:4px 12px">Training Epochs</td><td style="padding:4px 12px">{tr['epochs']}</td></tr>
<tr style="background:#f0f0f0"><td style="padding:4px 12px">Final train_loss / val_loss</td><td style="padding:4px 12px">{tr['final_train_loss']:.4f} / {tr['final_val_loss']:.4f}</td></tr>
</table>
"""
display(HTML(html))

# Training History
print("\n📈 Training Convergence (loss + nRMSE + LR schedule):")
display(IPImage(filename=str(COMPARE_DIR / "01_training_history.png"), width=900))

In [ ]:
# ============================================================
# 7-2) Scatter Plot: MGN Predicted vs FEM Ground Truth (전체 테스트 노드)
# ============================================================
from IPython.display import display, Image as IPImage
from pathlib import Path

COMPARE_DIR = Path(r"D:\KDH\NvidiaNemo\mgn_fem_comparison")

print("📊 Scatter: Predicted vs True (1.2M+ test nodes)")
display(IPImage(filename=str(COMPARE_DIR / "02_scatter_all_test.png"), width=700))

In [ ]:
# ============================================================
# 7-3) Error Histogram + Per-sample nRMSE Distribution
# ============================================================
from IPython.display import display, Image as IPImage
from pathlib import Path

COMPARE_DIR = Path(r"D:\KDH\NvidiaNemo\mgn_fem_comparison")

print("📊 Error Distribution (전체 노드) + Per-sample nRMSE")
display(IPImage(filename=str(COMPARE_DIR / "03_error_histogram.png"), width=900))

In [ ]:
# ============================================================
# 7-4) Field Comparison: FEM | MGN | Error (6 대표 샘플 — Best~Worst percentiles)
# ============================================================
from IPython.display import display, Image as IPImage
from pathlib import Path

COMPARE_DIR = Path(r"D:\KDH\NvidiaNemo\mgn_fem_comparison")

print("📊 Field Comparison (Best / 25th / Median / 75th / 90th / Worst)")
display(IPImage(filename=str(COMPARE_DIR / "04_field_comparison.png"), width=1000))

In [ ]:
# ============================================================
# 7-5) Per-sample R² 및 nRMSE 분포 (180 test samples)
# ============================================================
from IPython.display import display, Image as IPImage
from pathlib import Path

COMPARE_DIR = Path(r"D:\KDH\NvidiaNemo\mgn_fem_comparison")

print("📊 Per-sample R² (sorted) + nRMSE (sorted)")
display(IPImage(filename=str(COMPARE_DIR / "05_per_sample_metrics.png"), width=900))

# 7-6) 물리 단위 비교: Tesla [T] + 상대오차 [%]

`NormaliseTransform`은 **입력 x (material_id)만 정규화**하고 **타겟 y (Bnorm)는 원본 Tesla 그대로** 유지합니다.  
따라서 FEM/MGN 예측값은 이미 **Tesla** 단위이며, 아래 플롯은 이를 명시적으로 표시합니다.

| Metric | Value |
|--------|-------|
| RMSE | **0.2758 T** |
| nRMSE | **9.66%** |
| MAE | **0.1894 T** |
| Max Error | **2.0332 T** |
| R² | **0.7713** |
| Relative Error (median, \|B\|>10mT) | **18.2%** |

In [ ]:
# ============================================================
# 7-6a) Summary Table (Physical Units) + Training History
# ============================================================
import json
from pathlib import Path
from IPython.display import display, Image as IPImage, HTML

COMPARE_DIR_PHYS = Path(r"D:\KDH\NvidiaNemo\mgn_fem_comparison_physical")

with open(COMPARE_DIR_PHYS / "test_summary_physical.json") as f:
    summary_p = json.load(f)

tm = summary_p["total_metrics"]
ps = summary_p["per_sample_summary"]

html = f"""
<h3>MGN (EncoderProcessorDecoder) — Test Results in Physical Units</h3>
<table style="border-collapse:collapse; font-size:14px;">
<tr style="background:#e8f5e9"><th style="padding:6px 12px; text-align:left">Metric</th><th style="padding:6px 12px">Value</th><th style="padding:6px 12px">Unit</th></tr>
<tr><td style="padding:4px 12px">Test Samples</td><td style="padding:4px 12px"><b>{summary_p['test_samples']}</b> ({summary_p['total_nodes']:,} nodes)</td><td></td></tr>
<tr style="background:#fff3cd"><td style="padding:4px 12px"><b>RMSE</b></td><td style="padding:4px 12px"><b>{tm['RMSE_T']:.4f}</b></td><td style="padding:4px 12px"><b>T</b></td></tr>
<tr style="background:#fff3cd"><td style="padding:4px 12px"><b>nRMSE</b></td><td style="padding:4px 12px"><b>{tm['nRMSE_pct']:.2f}</b></td><td style="padding:4px 12px"><b>%</b></td></tr>
<tr style="background:#d4edda"><td style="padding:4px 12px"><b>R²</b></td><td style="padding:4px 12px"><b>{tm['R2']:.4f}</b></td><td></td></tr>
<tr><td style="padding:4px 12px">MAE</td><td style="padding:4px 12px">{tm['MAE_T']:.4f}</td><td style="padding:4px 12px">T</td></tr>
<tr><td style="padding:4px 12px">Max Error</td><td style="padding:4px 12px">{tm['MaxErr_T']:.4f}</td><td style="padding:4px 12px">T</td></tr>
<tr style="background:#f8d7da"><td style="padding:4px 12px">Relative Error (median, |B|&gt;10mT)</td><td style="padding:4px 12px">{tm['Relative_Error_median_pct']:.1f}</td><td style="padding:4px 12px">%</td></tr>
<tr style="background:#f8d7da"><td style="padding:4px 12px">Relative Error (mean, |B|&gt;10mT)</td><td style="padding:4px 12px">{tm['Relative_Error_mean_pct']:.1f}</td><td style="padding:4px 12px">%</td></tr>
</table>
<br><p style="font-size:12px; color:#666;">Note: NormaliseTransform only normalizes input x (material_id with μ=47.85, σ=33.89). 
Target y (Bnorm) is <b>NOT normalized</b> — values are raw Tesla from FEA.</p>
"""
display(HTML(html))

print("\n📈 Training History:")
display(IPImage(filename=str(COMPARE_DIR_PHYS / "01_training_history.png"), width=900))

In [ ]:
# ============================================================
# 7-6b) Scatter Plot — Bnorm [T]
# ============================================================
from IPython.display import display, Image as IPImage
from pathlib import Path

COMPARE_DIR_PHYS = Path(r"D:\KDH\NvidiaNemo\mgn_fem_comparison_physical")

print("📊 Scatter: Predicted vs True [Tesla]  (1.2M+ test nodes)")
display(IPImage(filename=str(COMPARE_DIR_PHYS / "02_scatter_tesla.png"), width=700))

In [ ]:
# ============================================================
# 7-6c) Error Histogram — Absolute [T] + nRMSE [%] + Relative Error [%]
# ============================================================
from IPython.display import display, Image as IPImage
from pathlib import Path

COMPARE_DIR_PHYS = Path(r"D:\KDH\NvidiaNemo\mgn_fem_comparison_physical")

print("📊 Error Distributions: Absolute [T] | nRMSE [%] | Relative Error [%]")
display(IPImage(filename=str(COMPARE_DIR_PHYS / "03_error_histogram_tesla.png"), width=1000))

In [ ]:
# ============================================================
# 7-6d) Field Comparison — FEM [T] | MGN [T] | Error [T] | Relative Error [%]
# ============================================================
from IPython.display import display, Image as IPImage
from pathlib import Path

COMPARE_DIR_PHYS = Path(r"D:\KDH\NvidiaNemo\mgn_fem_comparison_physical")

print("📊 4-column Field Comparison: FEM [T] | MGN Pred [T] | Error [T] | Relative Error [%]")
print("   (Best → Worst percentiles, 6 representative samples)")
display(IPImage(filename=str(COMPARE_DIR_PHYS / "04_field_comparison_tesla.png"), width=1100))

In [ ]:
# ============================================================
# 7-6e) Per-Sample Metrics — R², nRMSE [%], RMSE [T], Relative Error [%]
# ============================================================
from IPython.display import display, Image as IPImage
from pathlib import Path

COMPARE_DIR_PHYS = Path(r"D:\KDH\NvidiaNemo\mgn_fem_comparison_physical")

print("📊 Per-Sample Metrics: R² | nRMSE [%] | RMSE [T] | Relative Error [%]")
display(IPImage(filename=str(COMPARE_DIR_PHYS / "05_per_sample_metrics_tesla.png"), width=950))

# 8) Unified AI Model Comparison (Shared Utility)
아래 셀은 `model_compare_viz.py` 공통 모듈을 사용해 GT 대비 모델별 추론 결과를 동일 형식으로 시각화합니다.

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt

from model_compare_viz import (
    load_node_comparison_npz,
    load_mesh_triangles_from_h5,
    plot_model_comparison_grid,
)

BASE_DIR = Path("D:/KDH/NvidiaNemo")
bundle = load_node_comparison_npz(BASE_DIR / "field_compare_nodes_allsteps.npz")
mesh_triangles = load_mesh_triangles_from_h5(BASE_DIR, bundle["case_idx"])

step = min(20, bundle["n_steps"] - 1)
out_png = BASE_DIR / "paper_figures" / f"model_compare_step{step:03d}_from_pyMCAD4SolverX.png"

fig = plot_model_comparison_grid(
    bundle["node_x_all"],
    bundle["node_y_all"],
    bundle["data_dict"],
    step=step,
    quantity="bmag",
    models=("MGN", "FNO", "GINO", "RNN"),
    mesh_triangles=mesh_triangles,
    save_path=out_png,
)
print(f"saved: {out_png}")
plt.show()